# I. Sequence-based features

In [18]:
from Bio import SeqIO
from collections import Counter
from propy import PyPro

def calculate_aac(sequence: str) -> dict:
    """Calculate AAC with all 20 standard amino acids"""
    standard_amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
    aac = Counter(sequence)
    total_length = len(sequence)

    # Создаем словарь со всеми аминокислотами, включая отсутствующие
    result = {}
    for aa in standard_amino_acids:
        result[aa] = aac.get(aa, 0) / total_length

    return result

def calculate_dpc(sequence: str) -> dict:
    """Calculate DPC with all possible dipeptides"""
    standard_amino_acids = list('ACDEFGHIKLMNPQRSTVWY')

    # Создаем все возможные дипептиды
    all_dipeptides = [a + b for a in standard_amino_acids for b in standard_amino_acids]

    dpc = {}
    total_pairs = len(sequence) - 1

    # Считаем частоты для существующих дипептидов
    for i in range(len(sequence) - 1):
        dipeptide = sequence[i:i+2]
        dpc[dipeptide] = dpc.get(dipeptide, 0) + 1

    # Создаем полный словарь со всеми дипептидами
    result = {}
    for dp in all_dipeptides:
        result[dp] = dpc.get(dp, 0) / total_pairs if total_pairs > 0 else 0

    return result

def calculate_qso(sequence, max_lag=30, weight=0.1):
    """Calculate Quasi-sequence-order descriptors"""
    return PyPro.GetProDes(sequence).GetQSO(maxlag=max_lag, weight=weight)

# II. Physicochemical features

In [ ]:
import numpy as np

physicochemical_groups: dict[str, dict[str, int]] = {
    'hydrophobicity': {
        'A': 1, 'R': -1, 'N': -1, 'D': -1, 'C': 1, 'Q': -1,
        'E': -1, 'G': 0, 'H': -1, 'I': 1, 'L': 1, 'K': -1,
        'M': 1, 'F': 1, 'P': 0, 'S': -1, 'T': -1, 'W': 1,
        'Y': -1, 'V': 1
    },
    'polarity': {
        'A': 0, 'R': 1, 'N': 1, 'D': 1, 'C': 0, 'Q': 1,
        'E': 1, 'G': 0, 'H': 1, 'I': 0, 'L': 0, 'K': 1,
        'M': 0, 'F': 0, 'P': 0, 'S': 1, 'T': 1, 'W': 0,
        'Y': 1, 'V': 0
    },
}


def calculate_physicochemical(sequence, properties):
    """Calculate physicochemical properties composition"""
    results = {}
    for prop_name, prop_dict in properties.items():
        prop_values = [prop_dict.get(aa, 0) for aa in sequence]
        results[prop_name] = {
            'mean': np.mean(prop_values),
            'std': np.std(prop_values),
            'composition': Counter(prop_values)
        }
    return results


def calculate_ctdt(sequence, property_dict):
    """Calculate Composition/Transition/Distribution descriptors"""
    # Convert sequence to property values
    prop_values = [property_dict.get(aa, 0) for aa in sequence]

    # Composition: percentage of each class
    composition = Counter(prop_values)
    total = len(sequence)
    comp_result = {
        f"comp_{cls}": count/total for cls, count in composition.items()
    }

    # Transition: transitions between classes
    transition_result = {}
    for i in range(len(prop_values) - 1):
        cls_pair = tuple(sorted([prop_values[i], prop_values[i+1]]))
        if cls_pair[0] != cls_pair[1]:  # Only count transitions between different classes
            transition_result[cls_pair] = transition_result.get(cls_pair, 0) + 1

    # Normalize transitions
    total_transitions = len(sequence) - 1
    trans_result = {f"trans_{cls1}_{cls2}": count/total_transitions
                   for (cls1, cls2), count in transition_result.items()}

    # Distribution: distribution of property values along sequence
    # This would typically be calculated as 5-point distribution (0%, 25%, 50%, 75%, 100%)
    return {**comp_result, **trans_result}

In [ ]:
import csv

def flatten_dict(d, parent_key='', sep='_'):
    """
    Преобразует вложенный словарь в плоский словарь с объединенными ключами
    """
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        elif isinstance(v, list):
            # Для списков создаем отдельные колонки с индексами
            for i, item in enumerate(v):
                items.append((f"{new_key}{sep}{i}", item))
        else:
            items.append((new_key, v))
    return dict(items)

def features_to_csv(features_list, output_filename):
    """
    Записывает все извлеченные признаки в CSV файл

    Args:
        features_list: список словарей с признаками для каждой последовательности
        output_filename: имя выходного CSV файла
    """
    if not features_list:
        return

    # Преобразуем все вложенные словари в плоские
    flattened_features = []
    for feature_dict in features_list:
        flattened = flatten_dict(feature_dict)
        flattened_features.append(flattened)

    # Получаем все уникальные ключи (названия столбцов)
    all_keys = set()
    for flattened in flattened_features:
        all_keys.update(flattened.keys())

    # Сортируем ключи для единообразия
    sorted_keys = sorted(all_keys)

    # Записываем в CSV
    with open(output_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=sorted_keys)
        writer.writeheader()

        for flattened in flattened_features:
            # Заполняем отсутствующие значения пустыми строками
            row = {key: flattened.get(key, '') for key in sorted_keys}
            writer.writerow(row)

    print(f"Признаки успешно записаны в {output_filename}")

# Извлечение признаков из датасета в файл csv

In [22]:
path_train_neg = "Datasets/T6SE_Training_Neg_1112.fasta"
path_train_pos = "Datasets/T6SE_Training_Pos_138.fasta"

In [ ]:
def get_label(id: str) -> int:
    if "Non-effector" in id:
        return 0
    else:
        return 1

In [93]:
def record_features(path, label=None) -> None:
    all_features = []
    for record in SeqIO.parse(path, "fasta"):
        seq_str = str(record.seq)
        features = {
            'id': record.id,
            'aac': calculate_aac(seq_str),
            'dpc': calculate_dpc(seq_str),
            'qso': calculate_qso(seq_str),
            'physicochemical': calculate_physicochemical(seq_str, physicochemical_groups),
            'ctdt': {
                prop: calculate_ctdt(seq_str, prop_dict) for prop, prop_dict in physicochemical_groups.items()
            },
            'label': get_label(record.id) if label is None else label
        }
        all_features.append(features)
    features_to_csv(all_features, f"{path}.csv")

In [94]:
record_features('Datasets/Test.fasta')
record_features(path_train_pos, 1)
record_features(path_train_neg, 0)

Признаки успешно записаны в Datasets/Test.fasta.csv
Признаки успешно записаны в Datasets/T6SE_Training_Pos_138.fasta.csv
Признаки успешно записаны в Datasets/T6SE_Training_Neg_1112.fasta.csv


# Построение классификатора

In [1]:
import pandas as pd

def read_data_csv(path):
    return pd.read_csv(
		path,
		sep=',',
		encoding='utf-8',
		header=0
	)

def check_data(dataframe):
	print("Есть ли пустые значения в данных:", dataframe.isnull().values.any())

	print("Количество пустых значений по столбцам:")
	print(dataframe.isnull().sum())

	print("Доля пустых значений по столбцам:")
	print(dataframe.isnull().mean())

	print("Информация о DataFrame:")
	dataframe.info()

In [2]:
positive_df = read_data_csv('Datasets/T6SE_Training_Pos_138.fasta.csv')
negative_df = read_data_csv('Datasets/T6SE_Training_Neg_1112.fasta.csv')
test_df = read_data_csv('Datasets/Test.fasta.csv')

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [4]:
# Объединение датасетов
df = pd.concat([positive_df, negative_df], axis=0, ignore_index=True)
df = df.drop('id', axis=1)

# Разделение на признаки и целевую переменную
X = df.drop('label', axis=1)
Y = df['label']

# Проверка размерности
print(f"Размерность данных: {X.shape}")
print(f"Количество положительных примеров: {sum(Y == 1)}")
print(f"Количество отрицательных примеров: {sum(Y == 0)}")

Размерность данных: (1250, 538)
Количество положительных примеров: 138
Количество отрицательных примеров: 1112


In [5]:
# Для борьбы с дисбалансом классов используем технику взвешивания классов
from sklearn.utils import class_weight

# Вычисление весов классов
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(Y),
    y=Y
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"Веса классов: {class_weight_dict}")

Веса классов: {0: np.float64(0.5620503597122302), 1: np.float64(4.528985507246377)}


In [6]:
# Разделение на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# Масштабирование признаков (особенно важно для SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Тренировочная выборка: {X_train_scaled.shape}")
print(f"Тестовая выборка: {X_test_scaled.shape}")

Тренировочная выборка: (1000, 538)
Тестовая выборка: (250, 538)


In [7]:
# Создание и обучение SVM модели
svm_model = SVC(
    kernel='rbf',  # Радиальная базисная функция
    C=1.0,         # Параметр регуляризации
    gamma='scale', # Коэффициент для RBF ядра
    class_weight=class_weight_dict,  # Балансировка классов
    probability=True,  # Для получения вероятностей
    random_state=42
)

# Обучение модели
svm_model.fit(X_train, y_train)

# Предсказания
y_pred_svm = svm_model.predict(X_test_scaled)
y_pred_proba_svm = svm_model.predict_proba(X_test_scaled)

# Оценка модели
print("=== SVM Модель ===")
print(f"Точность: {accuracy_score(y_test, y_pred_svm):.4f}")
print("\nМатрица ошибок:")
print(confusion_matrix(y_test, y_pred_svm))
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_svm))

=== SVM Модель ===
Точность: 0.1120

Матрица ошибок:
[[  0 222]
 [  0  28]]

Отчет по классификации:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       222
           1       0.11      1.00      0.20        28

    accuracy                           0.11       250
   macro avg       0.06      0.50      0.10       250
weighted avg       0.01      0.11      0.02       250



e:\BioInformatics\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
e:\BioInformatics\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
e:\BioInformatics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\BioInformatics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh

In [8]:
# Создание и обучение Random Forest модели
rf_model = RandomForestClassifier(
    n_estimators=100,      # Количество деревьев
    max_depth=None,        # Максимальная глубина деревьев
    min_samples_split=2,   # Минимальное количество samples для разделения
    min_samples_leaf=1,    # Минимальное количество samples в листе
    class_weight=class_weight_dict,  # Балансировка классов
    random_state=42,
    n_jobs=-1  # Использование всех процессоров
)

# Обучение модели
rf_model.fit(X_train, y_train)  # Для RF масштабирование не обязательно

# Предсказания
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)

# Оценка модели
print("=== Random Forest Модель ===")
print(f"Точность: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nМатрица ошибок:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_rf))

=== Random Forest Модель ===
Точность: 0.9080

Матрица ошибок:
[[221   1]
 [ 22   6]]

Отчет по классификации:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95       222
           1       0.86      0.21      0.34        28

    accuracy                           0.91       250
   macro avg       0.88      0.60      0.65       250
weighted avg       0.90      0.91      0.88       250



# Классификатор на torch

In [48]:
import torch.nn as nn

class BioClassifier(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.act = nn.Linear(input_size, 269)
        self.bn = nn.BatchNorm1d(269)
        self.fn = nn.ReLU()
        self.act1 = nn.Linear(269, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fn1 = nn.ReLU()
        self.act2 = nn.Linear(64, 16)
        self.bn2 = nn.BatchNorm1d(16)
        self.fn2 = nn.ReLU()
        self.act3 = nn.Linear(16, 2)
        # self.classifier = nn.Sigmoid()
    
    def forward(self, x):
        x = self.fn(self.bn(self.act(x)))
        x = self.fn1(self.bn1(self.act1(x)))
        x = self.fn2(self.bn2(self.act2(x)))
        # x = self.classifier(self.act3(x))
        x = self.act3(x)
        return x

In [49]:
# Инициализация модели
input_size = X_train_scaled.shape[1]
model = BioClassifier(input_size=input_size)

print(model)

BioClassifier(
  (act): Linear(in_features=538, out_features=269, bias=True)
  (bn): BatchNorm1d(269, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fn): ReLU()
  (act1): Linear(in_features=269, out_features=64, bias=True)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fn1): ReLU()
  (act2): Linear(in_features=64, out_features=16, bias=True)
  (bn2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fn2): ReLU()
  (act3): Linear(in_features=16, out_features=2, bias=True)
)


In [50]:
import torch

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float)
y_train_t = torch.tensor(y_train, dtype=torch.float).reshape(-1, 1)

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float)
y_test_t = torch.tensor(list(y_test), dtype=torch.float).reshape(-1, 1)

In [55]:
import torch
import torch.optim as optim

# Обучение модели
loss_fn = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float)
)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

n_epochs = 100
batch_size = 32
patience = 3
best_epoch, old_loss = 0, 1e100

for epoch in range(n_epochs):
    for i in range(0, len(X_train_t), batch_size):
        Xbatch = X_train_t[i:i+batch_size]
        y_pred = model(Xbatch)
        ybatch = y_train_t[i:i+batch_size]
        ybatch = torch.argmax(ybatch, dim=1)
        loss = loss_fn(y_pred, ybatch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    loss = 0.0
    n = 0
    for i in range(0, len(X_test_t), batch_size):
        Xbatch = X_test_t[i:i+batch_size]
        y_pred = model(Xbatch)
        ybatch = y_test_t[i:i+batch_size]
        ybatch = torch.argmax(ybatch, dim=1)
        loss += loss_fn(y_pred, ybatch)
        n += 1
    loss = loss / n
    
    if loss < old_loss:
            best_epoch = epoch
            old_loss = loss
            best_model = model
    if epoch > best_epoch + patience:
            break

    print(f'Finished epoch {epoch}, latest loss {loss}')

Finished epoch 0, latest loss 0.7056325674057007
Finished epoch 1, latest loss 0.6778595447540283
Finished epoch 2, latest loss 0.6523696184158325
Finished epoch 3, latest loss 0.6296027302742004
Finished epoch 4, latest loss 0.6077632308006287
Finished epoch 5, latest loss 0.5873844027519226
Finished epoch 6, latest loss 0.5678662061691284
Finished epoch 7, latest loss 0.5483600497245789
Finished epoch 8, latest loss 0.5305995345115662
Finished epoch 9, latest loss 0.5125128030776978
Finished epoch 10, latest loss 0.4964539110660553
Finished epoch 11, latest loss 0.4799530506134033
Finished epoch 12, latest loss 0.4649420976638794
Finished epoch 13, latest loss 0.45057913661003113
Finished epoch 14, latest loss 0.436086505651474
Finished epoch 15, latest loss 0.4229656159877777
Finished epoch 16, latest loss 0.40916022658348083
Finished epoch 17, latest loss 0.3967752754688263
Finished epoch 18, latest loss 0.38552308082580566
Finished epoch 19, latest loss 0.3736763000488281
Finished

In [56]:
yp = best_model(X_test_t).detach().numpy()
yp = np.argmax(yp, axis=1)

print(yp.shape, y_test.shape)
print(yp[0], y_test_t[0])

(250,) (250,)
0 tensor([0.])


In [57]:
from sklearn.metrics import confusion_matrix, classification_report
print(classification_report(y_test_t, yp))
print(confusion_matrix(y_test_t, yp))

              precision    recall  f1-score   support

         0.0       0.89      1.00      0.94       222
         1.0       0.00      0.00      0.00        28

    accuracy                           0.89       250
   macro avg       0.44      0.50      0.47       250
weighted avg       0.79      0.89      0.84       250

[[222   0]
 [ 28   0]]


e:\BioInformatics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\BioInformatics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\BioInformatics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# Кросс-валидация и подбор гиперпараметров

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

# Кросс-валидация для SVM
svm_scores = cross_val_score(svm_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"SVM Кросс-валидация (средняя точность): {svm_scores.mean():.4f}")

# Кросс-валидация для Random Forest
rf_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"RF Кросс-валидация (средняя точность): {rf_scores.mean():.4f}")

# Подбор гиперпараметров для Random Forest
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_search_rf = GridSearchCV(
    RandomForestClassifier(class_weight=class_weight_dict, random_state=42),
    param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)
print(f"Лучшие параметры RF: {grid_search_rf.best_params_}")
print(f"Лучшая точность RF: {grid_search_rf.best_score_:.4f}")

SVM Кросс-валидация (средняя точность): 0.9280 (+/- 0.0196)
RF Кросс-валидация (средняя точность): 0.9020 (+/- 0.0080)
Лучшие параметры RF: {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 50}
Лучшая точность RF: 0.9170


In [ ]:
import joblib

In [ ]:
# Сохранение модели Random Forest
joblib.dump(rf_model, 'random_forest_model.pkl')
#Сохранение модели SVM
joblib.dump(svm_model, "svm_model.pkl")

['svm_model.pkl']

In [58]:
torch.save({
	'model_state_dict': model.state_dict(),
	'scaler': scaler,
	'input_size': X.shape[1]
}, f'torch_model_{92}%.pth')

# Тестируем качество классификатора на наборе DeepSecE

In [59]:
test = read_data_csv('Datasets/Test.fasta.csv')
test = test.drop('id', axis=1)

test_X = test.drop('label', axis=1)
test_Y = test['label']

In [62]:
# Загрузка с помощью joblib
loaded_model_rf = joblib.load('random_forest_model.pkl')
loaded_model_svm = joblib.load('svm_model.pkl')

In [18]:
# Проверка работы загруженной модели
y_pred_loaded_rf = loaded_model_rf.predict(test_X)
y_pred_loaded_svm = loaded_model_svm.predict(test_X)

print("=== SVM Модель ===")
print(f"Точность: {accuracy_score(test_Y, y_pred_loaded_svm):.4f}")
print("\nМатрица ошибок:")
print(confusion_matrix(test_Y, y_pred_loaded_svm))
print("\nОтчет по классификации:")
print(classification_report(test_Y, y_pred_loaded_svm))

print("=== Random Forest Модель ===")
print(f"Точность: {accuracy_score(test_Y, y_pred_loaded_rf):.4f}")
print("\nМатрица ошибок:")
print(confusion_matrix(test_Y, y_pred_loaded_rf))
print("\nОтчет по классификации:")
print(classification_report(test_Y, y_pred_loaded_rf))

=== SVM Модель ===
Точность: 0.6919

Матрица ошибок:
[[1080  497]
 [  60  171]]

Отчет по классификации:
              precision    recall  f1-score   support

           0       0.95      0.68      0.79      1577
           1       0.26      0.74      0.38       231

    accuracy                           0.69      1808
   macro avg       0.60      0.71      0.59      1808
weighted avg       0.86      0.69      0.74      1808

=== Random Forest Модель ===
Точность: 0.9270

Матрица ошибок:
[[1576    1]
 [ 131  100]]

Отчет по классификации:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96      1577
           1       0.99      0.43      0.60       231

    accuracy                           0.93      1808
   macro avg       0.96      0.72      0.78      1808
weighted avg       0.93      0.93      0.91      1808



In [64]:
checkpoint = torch.load('torch_model_92%.pth', weights_only=False)

loaded_model_t = BioClassifier(checkpoint['input_size'])
loaded_model_t.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [65]:
y_pred_t = loaded_model_t(torch.tensor(test_X.to_numpy(), dtype=torch.float))
yp = np.argmax(y_pred_t.detach().numpy(), axis=1)
print(test_Y.shape, yp.shape)

(1808,) (1808,)


In [66]:
from sklearn.metrics import confusion_matrix, classification_report
print(classification_report(test_Y, yp))
print(confusion_matrix(test_Y, yp))

              precision    recall  f1-score   support

           0       0.87      1.00      0.93      1577
           1       0.00      0.00      0.00       231

    accuracy                           0.87      1808
   macro avg       0.44      0.50      0.47      1808
weighted avg       0.76      0.87      0.81      1808

[[1577    0]
 [ 231    0]]


e:\BioInformatics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\BioInformatics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\BioInformatics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
